In [ ]:
# setup
import os
ds_size = 50
embed_models = [
    "all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "nomic-ai/nomic-embed-text-v1.5"
    ]
embed_models_dirnames = []
for name in embed_models:
    embed_models_dirnames.append(name.split("/")[-1])
ds_pos_results = []
cache_dir = os.path.join(os.getcwd(), "data", "hf-source")

# Acquiring the Source Dataset
Using Huggingface's Datasets we can easily load the wikipedia (20231101.en) dataset. Since we are focusing on encyclopedic data, this will serve as the base for testing the dataset generation pipeline.

In [ ]:
from datasets import load_dataset

cache_dir = os.path.join(os.getcwd(), "data", "hf-source")
ds = load_dataset("wikimedia/wikipedia", "20231101.en", cache_dir=cache_dir)

In [ ]:
# verification (optional)
print(f"Dataset length: {len(ds['train'])}")
print(ds['train'][0])
print(ds['train'][-1])

In [ ]:
# create subset for dev
ds_shuffled = ds.shuffle(seed=97)
if ds_size > 0:
    ds_subset = ds_shuffled['train'].select(range(ds_size))
else:
    ds_subset = ds_shuffled['train']
print(f"Test subset length: {len(ds_subset)}")
print(f"Sample entry: {ds_subset[0]['title']}")

# Embedding Generation

In [ ]:
from sentence_transformers import SentenceTransformer

for model in embed_models:
    dirname = model.split("/")[-1]
    ds_embed_dir = os.path.join(os.getcwd(), "data", "ds_embed", dirname, str(ds_size))

    # generate embeddings
    model = SentenceTransformer(model, trust_remote_code=True)
    texts = [entry['text'] for entry in ds_subset]
    print(f"Generating embeddings for {len(texts)} entries...")
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=1)
    print(f"Generated embeddings with shape: {embeddings.shape}")
    ds_embed = ds_subset.add_column("embeddings", embeddings.tolist())
    print(f"Dataset columns: {ds_embed.column_names}")

    # save ds with embeddings
    print(f"Saving dataset with embeddings to: {ds_embed_dir}")
    ds_embed.save_to_disk(ds_embed_dir)
    print("Dataset saved successfully!")

# Preprocessing


In [ ]:
import numpy as np
from numpy import ndarray
from datasets import Dataset, DatasetDict
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA


def prep(data: Dataset | DatasetDict) -> ndarray:
    # L2-normalization
    embeddings_array = np.asarray(data["embeddings"], dtype=np.float32)
    embeddings_norm = normalize(embeddings_array, norm="l2", axis=1)

    # PCA red to 50 dims: De-noising and speedup
    pca = PCA(n_components=50, random_state=97, svd_solver="auto", whiten=False)
    data = pca.fit_transform(embeddings_norm)
    return data

# Pipeline A (UMAP)
Based on [`knoto`](https://github.com/whatphilipcodes/knoto), this pipeline uses `UMAP` to reduce `sBERT` embeddings into two (spatial) dimensions.

In [ ]:
import umap
from datasets import load_from_disk

for dirname in embed_models_dirnames:
    ds_embed_dir = os.path.join(os.getcwd(), "data", "ds_embed", dirname, str(ds_size))
    ds_pos_dir = os.path.join(os.getcwd(), "data", "ds_pos", dirname, "umap", str(ds_size))

    ds_embed = load_from_disk(ds_embed_dir)
    print(f"Loaded dataset with {len(ds_embed)} entries")

    # collect embeddings
    embeddings_array = prep(ds_embed['embeddings'])
    print(f"Embeddings shape: {embeddings_array.shape}")

    # apply UMAP reduction
    reducer = umap.UMAP(n_components=2, random_state=97, init="spectral")
    print("Applying UMAP reduction...")
    embed_reduced = reducer.fit_transform(embeddings_array)
    print(f"2D embeddings shape: {embed_reduced.shape}")

    # add to ds
    ds_pos = ds_embed.add_column("x", embed_reduced[:, 0].tolist())
    ds_pos = ds_pos.add_column("y", embed_reduced[:, 1].tolist())
    print(f"Final dataset columns: {ds_pos.column_names}")

    # save ds with positions
    ds_pos.save_to_disk(ds_pos_dir)
    ds_pos_results.append(ds_pos_dir)
    print("Dataset with UMAP positions saved successfully!")

# Pipeline B (t-SNE)
based on the `scikit-learn` implementation of `t-SNE`.

In [ ]:
from sklearn.manifold import TSNE
from datasets import load_from_disk

for dirname in embed_models_dirnames:
    ds_embed_dir = os.path.join(os.getcwd(), "data", "ds_embed", dirname, str(ds_size))
    ds_pos_dir = os.path.join(os.getcwd(), "data", "ds_pos", dirname, "tsne", str(ds_size))

    # load dataset with embeddings (reuse from UMAP pipeline)
    ds_embed = load_from_disk(ds_embed_dir)
    print(f"Loaded dataset with {len(ds_embed)} entries")

    # collect embeddings
    embeddings_array = prep(ds_embed['embeddings'])
    print(f"Embeddings shape: {embeddings_array.shape}")

    # apply t-SNE reduction
    tsne = TSNE(n_components=2, random_state=97, init="pca")
    print("Applying t-SNE reduction...")
    embed_reduced_tsne = tsne.fit_transform(embeddings_array)
    print(f"2D t-SNE embeddings shape: {embed_reduced_tsne.shape}")

    # add to dataset
    ds_pos_tsne = ds_embed.add_column("x", embed_reduced_tsne[:, 0].tolist())
    ds_pos_tsne = ds_pos_tsne.add_column("y", embed_reduced_tsne[:, 1].tolist())
    print(f"Final dataset columns: {ds_pos_tsne.column_names}")

    # save dataset with t-SNE positions
    ds_pos_tsne.save_to_disk(ds_pos_dir)
    ds_pos_results.append(ds_pos_dir)
    print("Dataset with t-SNE positions saved successfully!")

# Visualization

In [ ]:
from datasets import load_from_disk
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

img_dir = os.path.join(os.getcwd(), "img")
# Ensure img directory exists
os.makedirs(img_dir, exist_ok=True)

for dir in ds_pos_results:
    ds_pos = load_from_disk(dir)

    xs = ds_pos["x"]
    ys = ds_pos["y"]

    plt.figure()
    sns.scatterplot(
        x=xs,
        y=ys,
        s=10,
        alpha=0.7
    )
    # plt.axis("off")
    plt.tight_layout()
    
    # Generate filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    # Extract directory name parts for filename
    dir_parts = dir.replace(os.getcwd(), "").strip(os.sep).split(os.sep)
    # Create filename: embedding_model_reduction_method_size_timestamp
    filename = f"{dir_parts[2]}_{dir_parts[3]}_{dir_parts[4]}_{timestamp}.pdf"
    filepath = os.path.join(img_dir, filename)
    
    # Save the plot as PDF (vector format, best for LaTeX)
    plt.savefig(filepath, format='pdf', bbox_inches='tight')
    print(f"Saved plot to: {filepath}")
    
    plt.show()